In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds

!pip install tensorflow-datasets

In [ ]:
# List to store the IMDB training dataset
imdb_sentences_trainset = []

In [ ]:
# Load the training split of the dataset and convert tensors to numpy
train_data = tfds.as_numpy(tfds.load('imdb_reviews', split="train"))

In [ ]:
type(train_data)

-------------------------

### Optional Section: To dissect data further

In [ ]:
import numpy as np

data = list(train_data)  # Materialize to list

# See what each data in the dataset contains
print(data[1])
print("\n---------------\n")

X = np.stack([list(d.values())[1] for d in data])
y = np.array([list(d.values())[0] for d in data])

print(X[:3])
print("\n---------------\n")
print(y[:3])

#### End of Optional section

------------------------

In [ ]:
# Extract text reviews from the dataset
for item in train_data:
    imdb_sentences_trainset.append(str(item['text'].decode('utf-8'))) # Add the decode('utf-8') part to remove the bytestring character marker i.e. b"hello"

In [ ]:
len(imdb_sentences_trainset)

In [ ]:
# Print a few sentences to see how they look
for s in imdb_sentences_trainset[:5]:
    print(f"{s}\n-----\n")

### Remove HTML Tags

In [ ]:
from bs4 import BeautifulSoup

# Function to remove HTML tags
def remove_html_tags(sentence):
  # Create a BeautifulSoup object with the input sentence
  soup = BeautifulSoup(sentence, "lxml")
  # Extract and return the plain text without HTML tags
  return soup.get_text()

In [ ]:
trainset_html_tags_removed = []
for sentence in imdb_sentences_trainset:
    clean_sentence = remove_html_tags(sentence)
    trainset_html_tags_removed.append(clean_sentence)

# view just first five sentences to quickly get an overview of the results
for s in trainset_html_tags_removed[:5]:
    print(f"{s}\n")


### Tokenize Data

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [ ]:
tokenizer = Tokenizer(num_words = 5000) # num_words=100 limits vocabulary to top 100 most frequent words

tokenizer.fit_on_texts(trainset_html_tags_removed) # Analyzes the sentences to build the vocabulary

word_index_trainset = tokenizer.word_index #Retrieves the dictionary mapping each word to its assigned integer index. Words are ranked by frequency (most frequent = index 1).

# print(word_index_trainset)

### Stopwords Removal

In [ ]:
stop_words = ["the", "a", "of","and","to","this","for","in","but","is","it","that","was","with","as","an","by","at","has","from","who","or","there","priest"]

In [ ]:
def remove_stopwords(sentence):
  words = sentence.split()  # Split the sentence into words
  # print(words)
  filtered_sentence = ""    # Initialize an empty string for the filtered sentence
  for word in words:
    # If the word is not in the stopwords list, add it to the filtered sentence
    if word.strip().lower() not in stop_words:  # Convert word to lowercase for case-insensitive matching
      filtered_sentence += word + " "
  return filtered_sentence.strip()  # Return the filtered sentence without leading/trailing spaces

In [ ]:
trainset_sentences_without_stopwords = []
for sentence in trainset_html_tags_removed:
    stopwords_filtered_sentence = remove_stopwords(sentence)
    trainset_sentences_without_stopwords.append(stopwords_filtered_sentence)

In [ ]:
for s in trainset_sentences_without_stopwords[:5]:
    print(f"{s}\n-----\n")

#### Looks like some stopwords were not removed.

#### Cause:

Punctuation marks might be causing the issue. So, let's remove them all!

### Remove punctuation

In [ ]:
import string

def remove_punctuation(sentence):
    # Create a translation table to remove punctuation
    table = str.maketrans('', '', string.punctuation)

    ###################################################################
    ## If bytestring remains, you can use this strip to get rid of them
    # sentence = sentence.lstrip("b'").rstrip("'")
    ###################################################################

    # Split the sentence into words
    words = sentence.split()

    # Initialize an empty string for the filtered sentence
    filtered_sentence = ""

    # Iterate through words in the sentence
    for word in words:
      word = word.translate(table) # Remove punctuation from each word
      # If the word is not in stopwords, add it to the filtered sentence
      if word.lower() not in stop_words:  # Convert to lowercase for consistent matching
        filtered_sentence += word + " "
    return filtered_sentence.strip()

In [ ]:
trainset_sentences_without_punctuation = []
for sentence in trainset_html_tags_removed:
    trainset_sentences_without_punctuation.append(remove_punctuation(sentence))
for s in trainset_sentences_without_punctuation[:5]:
    print(f"{s}\n-----\n")

------------------------------

### Remove stopwords one more time

In [ ]:
trainset_sentences_without_stopwords = []
for sentence in trainset_sentences_without_punctuation:
    stopwords_filtered_sentence = remove_stopwords(sentence)
    trainset_sentences_without_stopwords.append(stopwords_filtered_sentence)

In [ ]:
len(trainset_sentences_without_stopwords)

In [ ]:
for s in trainset_sentences_without_stopwords[:5]:
    print(f"{s}\n-----\n")

### Tokenizing Again

In [ ]:
tokenizer = Tokenizer(num_words = 5000) # num_words=100 limits vocabulary to top 100 most frequent words

tokenizer.fit_on_texts(trainset_sentences_without_stopwords) # Analyzes the sentences to build the vocabulary

word_index = tokenizer.word_index #Retrieves the dictionary mapping each word to its assigned integer index. Words are ranked by frequency (most frequent = index 1).

# print(word_index)

In [ ]:
trainset_sequences = tokenizer.texts_to_sequences(trainset_sentences_without_stopwords)
for s in trainset_sequences[:5]:
    print(f"{s}\n-------\n")

### Padding

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
max_sequence_len = 200
trainset_padded = pad_sequences(trainset_sequences, maxlen=max_sequence_len)
for s in trainset_padded[:5]:
    print(f"{s}\n-------\n")

### Create Labels for TrainingSet

In [ ]:
# Extract labels for each reviews from the dataset
trainset_labels_imdb = []
for item in train_data:
    trainset_labels_imdb.append(int(item['label']))
trainset_labels = np.array(trainset_labels_imdb)

In [ ]:
print(trainset_labels[:5])

-------------------------

### Repeat the whole process for testset

#### Load Test dataset

In [ ]:
# List to store the IMDB testing dataset
imdb_sentences_testset = []

# Load the testing split of the dataset and convert tensors to numpy
test_data = tfds.as_numpy(tfds.load('imdb_reviews', split="test"))

# Extract text reviews from the dataset
for item in test_data:
    imdb_sentences_testset.append(str(item['text'].decode('utf-8'))) # Add the decode('utf-8') part to remove the bytestring character marker i.e. b"hello"

print(imdb_sentences_testset[:2])

#### Remove HTML Tags from test dataset

In [ ]:
# Remove HTML Tags
testset_html_tags_removed = []
for sentence in imdb_sentences_testset:
    clean_sentence = remove_html_tags(sentence)
    testset_html_tags_removed.append(clean_sentence)

#### Remove Punctuation from test dataset

In [ ]:
testset_sentences_without_punctuation = []
for sentence in testset_html_tags_removed:
    testset_sentences_without_punctuation.append(remove_punctuation(sentence))
for s in testset_sentences_without_punctuation[:2]:
    print(f"{s}\n-----\n")

#### Remove Stopwords from test dataset

In [ ]:
testset_sentences_without_stopwords = []
for sentence in testset_html_tags_removed:
    stopwords_filtered_sentence = remove_stopwords(sentence)
    testset_sentences_without_stopwords.append(stopwords_filtered_sentence)

#### Tokenize test dataset

#### Generate sentence sequence for test dataset

In [ ]:
testset_sequences = tokenizer.texts_to_sequences(testset_sentences_without_stopwords)

#### Pad sentence sequence for test dataset

In [ ]:
max_sequence_len = 200
testset_padded = pad_sequences(testset_sequences, maxlen=max_sequence_len)

In [ ]:
# Extract labels for each reviews from the dataset
testset_labels_imdb = []
for item in test_data:
    testset_labels_imdb.append(int(item['label']))
testset_labels = np.array(testset_labels_imdb)
print(testset_labels[:5])

------------------------------------------

### Build a Model

In [ ]:
#Build the model
model = tf.keras.Sequential([tf.keras.layers.Embedding(input_dim=25000, output_dim=128),
                             tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64, return_sequences=True)),
                             tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64)),
                             tf.keras.layers.Dense(64, activation='relu'),
                             tf.keras.layers.Dropout(0.5),tf.keras.layers.Dense(1,activation='sigmoid')])

In [ ]:
# Compile the model
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

In [ ]:
#train the model
history = model.fit(trainset_padded, trainset_labels, epochs=5, validation_data=(testset_padded, testset_labels), batch_size=64)

### Evaluate the Model

In [ ]:
# Evaluate the model
loss, accuracy = model.evaluate(testset_padded, testset_labels)
print(f"Test Accuracy: {accuracy * 100:.2f}%")

In [ ]:
np.unique(trainset_labels)

### Plotting training and validation accuracy/loss

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

# 1. Training and Validation Accuracy/Loss
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy plot
axes[0].plot(history.history['accuracy'], label='Train Accuracy', marker='o')
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy', marker='s')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Model Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss plot
axes[1].plot(history.history['loss'], label='Train Loss', marker='o')
axes[1].plot(history.history['val_loss'], label='Val Loss', marker='s')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('Model Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Confusion Matrix

In [ ]:
# Generate predictions
y_pred_probs = model.predict(testset_padded)
y_pred = (y_pred_probs > 0.5).astype(int).flatten()  # Binary classification
# For multi-class: y_pred = np.argmax(y_pred_probs, axis=1)


In [ ]:
# Compute confusion matrix
cm = confusion_matrix(testset_labels, y_pred)

In [ ]:
# Plot confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negative', 'Positive'],
            yticklabels=['Negative', 'Positive'],
            cbar_kws={'label': 'Count'})
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Quantitative metrics
print(classification_report(testset_labels, y_pred,
                           target_names=['Negative', 'Positive']))

In [ ]:
# Accuracy breakdown
print(f"\nTest Accuracy: {np.mean(y_pred == testset_labels):.4f}")